In [ ]:
import json

import matplotlib.pyplot as plt
from paths import checkpoint_dir
from rollout import latest_checkpoint, load_checkpoint

run = checkpoint_dir("m04-walk")                         # 开训时起的名字
rows = [json.loads(line) for line in (run / "metrics.jsonl").read_text().splitlines()]   # 每轮一行
len(rows), sorted(rows[-1])[:8]

In [ ]:
it = [r["iteration"] for r in rows]
plt.plot(it, [r["episode_reward"] for r in rows])        # 回合总奖励
plt.xlabel("iteration")
plt.ylabel("episode reward")

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, sharex=True)
top.plot(it, [r["Curriculum/action_rate_weight"] for r in rows])     # 动作变化惩罚的权重：分阶段加重
bottom.plot(it, [r["Episode_Termination/fell_over"] for r in rows])  # 摔倒
bottom.set_xlim(0, 2500)
bottom.set_xlabel("iteration")

In [ ]:
policy, ckpt = load_checkpoint(latest_checkpoint("m04-walk"), "cuda")
print(policy.actor)                                      # 61 → 512 → 256 → 128 → 14
sum(p.numel() for p in policy.actor.parameters())        # Actor 有多少个参数

In [ ]:
policy.std.detach().cpu().numpy().round(2)               # 14 个关节动作的标准差：训练时探索的幅度

In [ ]:
import hardware
from env import SKILLS
from rollout import evaluate

hw = hardware.load("2026-11-07-hd1910")
summary = evaluate(policy.act_inference, SKILLS["walk"], hw, ckpt["common_step_counter"], num_envs=64)
summary["falls"], summary["terms"]["Metrics/twist/error_vel_xy"]   # 摔了几次；平移速度跟踪误差（m/s）